# Nightingale: B0 and B1 on DDXPlus (EXP-003, EXP-004)

This notebook runs `scripts/train_baselines.py` on Colab, where the owner chose to run it
(decision D-7). It trains the prevalence baseline (B0), logistic regression and XGBoost (B1)
on DDXPlus's train split. It then scores them on the validate split, with the metrics of
`docs/05` and their 95% confidence intervals. The test split is never downloaded.

**Before you start:** the notebook asks for a GPU (T4); XGBoost uses it if it is there. If Colab
offers only a CPU, accept it: the run is just slower.

**Then:** *Runtime → Run all*. It takes about 20–30 minutes, most of it downloading and decoding
the 670 MB `train.csv`. At the end your browser downloads **`nightingale_b0_b1.zip`**.

**Keep this notebook and its outputs private.** Models trained on DDXPlus are never published
(`docs/11` §4). Nothing here needs a password or a token.

In [ ]:
# The commit Claude gave you. "master" works too: run.json records the exact commit either way.
COMMIT = "master"

In [ ]:
import os

if not os.path.exists("/content/nightingale"):
    !git clone -q https://github.com/HarshRohila02/nightingale.git /content/nightingale
%cd /content/nightingale
!git checkout -q {COMMIT}
!git log --oneline -1
# XGBoost 2.1 is the version on the owner's laptop, so the laptop can load the trained model.
!pip install -q "xgboost~=2.1" "scikit-learn~=1.5"

In [ ]:
# DDXPlus from Hugging Face, at the snapshot the laptop uses (docs/11 §4).
# test.csv is never fetched before Phase 4 (docs/05 §2).
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "train.csv", "validate.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir="data/raw/ddxplus",
    )
!ls -lh data/raw/ddxplus

In [ ]:
# Decode the vocabulary, then keep the 13 chest-pain conditions of each split (task 1a).
!python scripts/decode_ddxplus.py
!python scripts/build_ddxplus_chestpain.py --split train
!python scripts/build_ddxplus_chestpain.py --split validate

In [ ]:
import shutil

DEVICE = "cuda" if shutil.which("nvidia-smi") else "cpu"
print("XGBoost runs on", DEVICE)
!python scripts/train_baselines.py --device {DEVICE}

In [ ]:
# One zip to bring back: the models, the metrics, run.json, and both splits' summaries
# (the train counts re-confirm EXP-002). No patient rows are in it.
import shutil
from google.colab import files

for split in ("train", "validate"):
    shutil.copy(f"data/interim/ddxplus_chestpain_{split}.summary.json", "models/b0_b1/")
archive = shutil.make_archive("/content/nightingale_b0_b1", "zip", "models/b0_b1")
!ls -lh models/b0_b1 {archive}
files.download(archive)